The standard import statements supporting the code
- Additionally we used geopy -> **This is used to calculate the distance between the target and the source based on the latitude and longitude provided**
    

import pandas as pd
import numpy as np
from geopy.distance import geodesic
import matplotlib.pyplot as plt
import folium

In [34]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import matplotlib.pyplot as plt
import folium

In [35]:
RAW_DIRECTORY =  r"A:\JnJ_assignment_data_science\Data\Bronze"
STORAGE_DIRECTORY = r"A:\JnJ_assignment_data_science\Data\Silver"
employees_df = pd.read_csv(RAW_DIRECTORY + r"\Employees_table.csv")
Possible_stations_df = pd.read_csv(RAW_DIRECTORY + r"\Possible_Generated_stations.csv")


We have the target location which is J&J along with the latitude and longitude
- the function calculates the distance between target and source
- It is called and applied to the employees_df (which has the employees **lat** and **long**)

In [36]:
dest_location = {
    "Company" : "Johnson & Johnson Medical GmBH",
    "Location" : "Robert-Koch-Straße 1, 22851 Norderstedt",
    "lat":53.6907,
    "long":9.9806}

target = (dest_location["lat"],dest_location["long"])


def find_distance(tgt):
    emp_location = (tgt["Lat"],tgt["Long"])
    distance = round(geodesic(emp_location,target).km,2)
    return distance

In [37]:
employees_df["Distance_to_cover(kms)"] = employees_df.apply(find_distance,axis=1)

This function is to calculate the distance from the house location to the nearest station
- We have 2 variables 
    - one to store the nearest station
    - one to comapre the maximum location with the house -> nearest station and store the new distance in kms

- Finally we apply it to the dataframe (**employees_df**)

In [38]:
def nearest_to_living(emp_from_table,stations_df):
    loca = (emp_from_table["Lat"],emp_from_table["Long"])
    possible_nearest_station = None
    nearest_distance = float("inf")
    for idx,stations in stations_df.iterrows():
        station_locations = (stations["lat"],stations["long"])
        actual_distance = geodesic(loca,station_locations).km
        if actual_distance < nearest_distance:
            nearest_distance = actual_distance
            possible_nearest_station = stations["station"]
    
    return pd.Series({"Possible_nearest_station": possible_nearest_station,"distance_to_station(kms)":round(nearest_distance,2)})

In [39]:
stations_finalized = employees_df.apply(nearest_to_living,stations_df = Possible_stations_df,axis=1)
employees_df = pd.concat([employees_df, stations_finalized], axis=1)


This function simply calculates the time 
- base formulate distance  / speed


**Why use this function helps us to maintain a unified function so that we can reuse it in multiple timelines rather than doing the same operation again**



**Single responsibility principle**

In [40]:
def func_time(distance,speed):
    return round((distance / speed) * 60,1)

Few variable to be considered as hardcoded values because we need them, assuming we didnt take the actual hvv fahrplane and also we need variables to calculate time

- speed of the person assumed
- speed of transport which is public transport (bus preferably)
- speed of the train (very much required for time clustering)
- reasonable walk distance (We require this to have a conditional checks for people living close to the office)
- we need to keep in mind about the waiting time for bus and train (assumed)
- also we take the time for walking from the home to the station and transferring situations

In [41]:
ASSUMED_WALK_SPEED = 3         
ASSUMED_SPEED_OF_TRANSPORT  = 30     
ASSUMED_RAIL_SPEED = 35
MAX_REASONABLE_WALK_KM = 1
BUS_WAITING_TIME = 8 
TRAIN_WAITING_TIME = 7
WALKING_TIME = 10

This function helps to calculate the time taken from **Home -> Office**
- **Notice** : we have reused the func_time (which was created earlier)

In [42]:
def time_taken_from_home_to_office(dataframe):
    dataframe["home_to_office(minutes)"] = func_time(dataframe["Distance_to_cover(kms)"],ASSUMED_WALK_SPEED)
    return dataframe

employees_df = time_taken_from_home_to_office(employees_df)

We have 2 important functions
- add_column_station:
    - shows the time taken from the home to station (walkable distance we used the hardcoded variable for the speed)
    - shows the time taken from the home to station (if bus included, same variable for hardcoded speed value of the bus transport)

- optimality:
    - we have 2 condiitons here (because if the person if living very close to the station then we dont require any bus transport (**helps with avoiding waiting time and travel time by bus**). Similarly we must consider the other possibility too)
    
        - the distance from home to station is less than reasonable walking km(then bus is less possibility)
            - we add this time taken by walk to the final dataset
        - the distance form home to bus + we consider the waiting time for the bus
            - we add this time taken by bus(combined) back to the final dataset

In [43]:
def add_column_station(dataframe):
    dataframe["home_to_station(walk)"] = func_time(dataframe["distance_to_station(kms)"],ASSUMED_WALK_SPEED)
    dataframe["home_to_station(bus)"] = func_time(dataframe["distance_to_station(kms)"],ASSUMED_SPEED_OF_TRANSPORT)
    return dataframe
def optimality(dataset):
    if dataset["distance_to_station(kms)"] <= MAX_REASONABLE_WALK_KM:
        return dataset["home_to_station(walk)"]
    else:
        return dataset["home_to_station(bus)"] + BUS_WAITING_TIME
def add_optimal_station(dataframe):
    dataframe["home_to_station(minutes)"] = dataframe.apply(optimality,axis=1)
    return dataframe
    
employees_df = add_column_station(employees_df)
employees_df = add_optimal_station(employees_df)

This function is to calculate the distance between station to the office location

- we use the geodesic between the station and the target (returning in kms)

In [44]:
def station_to_workplace(stations):
    stat_location = (stations["lat"],stations["long"])
    distance = round(geodesic(stat_location,target).km,2)
    return distance

def adding_columns(stations_df):
    stations_df["station_to_workplace(kms)"] = stations_df.apply(station_to_workplace,axis=1)
    return stations_df

Possible_stations = adding_columns(Possible_stations_df)


We must also calculate the time taken for the distance covered by the length **station -> office location**

In [45]:
def add_time_to_reach(stations_df):
    stations_df["station_to_workplace(minutes)"] = func_time(stations_df["station_to_workplace(kms)"],ASSUMED_RAIL_SPEED)
    return stations_df

Possible_stations = add_time_to_reach(Possible_stations_df)

Now we have the available variables we perform a join

Notice we never added the distance and time to the employees_df table but **to the generated possibel stations dataframe**

- we perform a **left join** between *employees_df -> Possible_stations_df*
    - why because we want all the matching values from the right table to be merged with left (the parent table behaves like employees_df (which is the ultimate table))
    - the joining key is the station

In [46]:
employees_df = employees_df.merge( Possible_stations_df[
        [
            "station",
            "station_to_workplace(kms)",
            "station_to_workplace(minutes)"
        ]
    ],left_on="Possible_nearest_station",right_on="station",how="left"
)
employees_df.drop(columns=["station"],inplace=True)

Considering we have all the variables in place we perform the final processing for this table which is the summation

- we add the following:
    - home to station
    - **+**
    - station to workplace
    - **+**
    - waiting time for train *(assumed)*
    - **+**
    - walking time taken

This gives us the total time from **door - to - door** commute

In [47]:
def total_commute_time(dataframe):
    dataframe["total_time"] = dataframe["home_to_station(minutes)"] + dataframe["station_to_workplace(minutes)"] + TRAIN_WAITING_TIME + WALKING_TIME
    return dataframe    
employees_df = total_commute_time(employees_df)

**Improvisation**

I thought of a possibility

- if the total distance from the *home -> office* is < 1.5kms (*which i beleive mostly people may not choose the public transport but rather walk*)
- if true then we add the time taken from *home -> office* as the total time(special condition visible via **Map**)

In [48]:
def finalized_total_time(dataframe):
    if dataframe["Distance_to_cover(kms)"] <= 1.5:
        dataframe["total_time"] = dataframe["home_to_office(minutes)"]
    return dataframe

employees_df = employees_df.apply(finalized_total_time,axis=1)

In [49]:
employees_df.to_csv(STORAGE_DIRECTORY + r"\FE_cleaned_table.csv",index=False)